# TP N° 5 — Unidades 3 y 4
**Alumno:** Aramayo Pedro Mauricio Leandro

**Contenido**
1. Comparación de documentos con TF-IDF y similitud del coseno (RepA y RepB).
2. Resúmenes extractivos del discurso inaugural de Obama (2009) con dos métodos: frecuencia de palabras y TextRank.

> Para ejecutar en Colab: *Entorno de ejecución → Ejecutar todo*. No hace falta instalar nada: NLTK, scikit-learn y NetworkX ya vienen en Colab.

In [1]:
import nltk
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for recurso in ["punkt", "punkt_tab", "stopwords", "inaugural"]:
    nltk.download(recurso, quiet=True)

from nltk.corpus import stopwords, inaugural
print("Librerías listas")

Librerías listas


## 1) Comparación de documentos con TF-IDF

**Idea:** cada documento se convierte en un vector TF-IDF (peso alto para palabras frecuentes en el documento pero raras en el resto). Para buscar, la consulta se transforma con el mismo vocabulario y se mide la **similitud del coseno** contra cada documento del índice (1 = muy parecido, 0 = nada en común).

**Armado de cada representación**
- **RepA:** los 5 documentos del punto 3 del TP 1 → 4 forman el índice y 1 es la consulta.
- **RepB:** las 10 noticias del punto 2 del TP 3 → 9 forman el índice y 1 es la consulta.
- Dentro de cada grupo se usa siempre **la misma consulta** en las tres variantes.

**Variantes que pide la consigna**
| Variante | Configuración de `TfidfVectorizer` |
|---|---|
| a) Texto original | palabras sueltas, sin quitar nada (solo pasa a minúsculas) |
| b) Sin stop words | se eliminan palabras vacías del idioma (*el, de, que, the, of*…) |
| c) Bigramas | los términos son pares de palabras seguidas (*"banco central"*) |

### 1.1 Funciones comunes

In [2]:
def configurar_vectorizador(variante, idioma):
    """Devuelve el TfidfVectorizer de cada variante."""
    if variante == "a) Texto original":
        return TfidfVectorizer()
    if variante == "b) Sin stop words":
        return TfidfVectorizer(stop_words=stopwords.words(idioma))
    if variante == "c) Bigramas":
        return TfidfVectorizer(ngram_range=(2, 2))
    raise ValueError(variante)

VARIANTES = ["a) Texto original", "b) Sin stop words", "c) Bigramas"]


def buscar_top3(indice, consulta, variante, idioma):
    """Arma el TF-IDF con los n-1 documentos y devuelve los 3 más parecidos a la consulta."""
    vectorizador = configurar_vectorizador(variante, idioma)
    matriz = vectorizador.fit_transform(indice["texto"])      # representación de los n-1 documentos
    vector_consulta = vectorizador.transform([consulta])       # misma representación para la consulta
    puntajes = cosine_similarity(vector_consulta, matriz)[0]
    resultado = indice.assign(score=puntajes).sort_values("score", ascending=False).head(3)
    return resultado, len(vectorizador.vocabulary_)


def mostrar_resultados(nombre_rep, docs, nombre_consulta, idioma, largo_texto=400):
    """Corre las 3 variantes, imprime el top 3 con score y texto, y devuelve una tabla resumen."""
    indice = docs[docs["documento"] != nombre_consulta].reset_index(drop=True)
    consulta = docs.loc[docs["documento"] == nombre_consulta, "texto"].iloc[0]
    print(f"{nombre_rep}: índice de {len(indice)} documentos | consulta = {nombre_consulta}")
    print("Texto de la consulta:", consulta[:largo_texto], "...\n")

    filas = []
    for variante in VARIANTES:
        top3, tam_vocab = buscar_top3(indice, consulta, variante, idioma)
        print("=" * 90)
        print(f"{nombre_rep} — {variante}  (vocabulario: {tam_vocab} términos)")
        print("=" * 90)
        for puesto, (_, fila) in enumerate(top3.iterrows(), start=1):
            texto = fila["texto"].replace("\n", " ")
            print(f"{puesto}. {fila['documento']}  |  score = {fila['score']:.4f}")
            print("   ", texto[:largo_texto] + ("..." if len(texto) > largo_texto else ""))
            print()
            filas.append({"variante": variante, "puesto": puesto,
                          "documento": fila["documento"], "score": round(fila["score"], 4)})
    return pd.DataFrame(filas)


def tabla_comparativa(resumen):
    """Pone las tres variantes lado a lado: documento (score) por puesto."""
    resumen = resumen.assign(resultado=resumen["documento"] + " (" + resumen["score"].astype(str) + ")")
    return resumen.pivot(index="puesto", columns="variante", values="resultado")

### 1.2 Carga de los documentos

Los textos se leen de dos carpetas junto al cuaderno: `repA/` (5 archivos `.txt` del TP 1) y `repB/` (10 archivos `.txt` del TP 3). En Colab se pueden subir con el panel de archivos o con la celda de abajo.

In [3]:
import os, glob

def cargar_carpeta(carpeta):
    archivos = sorted(glob.glob(os.path.join(carpeta, "*.txt")))
    filas = []
    for ruta in archivos:
        with open(ruta, encoding="utf-8") as f:
            filas.append({"documento": os.path.basename(ruta), "texto": f.read().strip()})
    return pd.DataFrame(filas, columns=["documento", "texto"])

docs_A = cargar_carpeta("repA")
docs_B = cargar_carpeta("repB")
print("RepA:", len(docs_A), "documentos | RepB:", len(docs_B), "documentos")
if len(docs_A) != 5 or len(docs_B) != 10:
    print("ATENCIÓN: faltan los textos originales del TP 1 (5) y/o del TP 3 (10). "
          "Subilos a las carpetas repA/ y repB/ y volvé a ejecutar.")

RepA: 0 documentos | RepB: 0 documentos
ATENCIÓN: faltan los textos originales del TP 1 (5) y/o del TP 3 (10). Subilos a las carpetas repA/ y repB/ y volvé a ejecutar.


### 1.3 RepA — 5 documentos del TP 1 (índice de 4 + 1 consulta)

In [4]:
IDIOMA_A = "spanish"
CONSULTA_A = docs_A["documento"].iloc[-1] if len(docs_A) else None   # se reserva el último documento

if len(docs_A) == 5:
    resumen_A = mostrar_resultados("RepA", docs_A, CONSULTA_A, IDIOMA_A)
    display(tabla_comparativa(resumen_A))
else:
    print("RepA pendiente: faltan los 5 documentos del TP 1.")

RepA pendiente: faltan los 5 documentos del TP 1.


### 1.4 RepB — 10 noticias del TP 3 (índice de 9 + 1 consulta)

In [5]:
IDIOMA_B = "spanish"
CONSULTA_B = docs_B["documento"].iloc[-1] if len(docs_B) else None   # se reserva la última noticia

if len(docs_B) == 10:
    resumen_B = mostrar_resultados("RepB", docs_B, CONSULTA_B, IDIOMA_B)
    display(tabla_comparativa(resumen_B))
else:
    print("RepB pendiente: faltan las 10 noticias del TP 3.")

RepB pendiente: faltan las 10 noticias del TP 3.


### 1.5 ¿Se produce algún cambio en los resultados?

*(Se completa con los resultados reales una vez cargados los textos del TP 1 y del TP 3.)*

## 2) Resúmenes extractivos del discurso inaugural de Obama (2009)

Un resumen **extractivo** no redacta texto nuevo: elige las oraciones más importantes del original y las muestra en el orden en que aparecen. Se generan **dos resúmenes de 8 oraciones** (la consigna pide entre 7 y 10) con dos métodos distintos:

1. **Frecuencia de palabras:** las palabras que más se repiten (sin stop words) indican los temas centrales; cada oración suma la frecuencia normalizada de sus palabras.
2. **TextRank:** cada oración es un nodo de un grafo; dos oraciones se conectan según su similitud del coseno (TF-IDF). PageRank premia a las oraciones que se parecen a muchas otras, o sea, las más "centrales" del discurso.

Las oraciones se conservan en inglés, tal como están en el corpus.

In [6]:
texto_obama = inaugural.raw("2009-Obama.txt")
oraciones = [o.replace("\n", " ").strip() for o in nltk.sent_tokenize(texto_obama)]
stop_en = set(stopwords.words("english"))

def palabras_de_contenido(oracion):
    return [p.lower() for p in nltk.word_tokenize(oracion) if p.isalpha() and p.lower() not in stop_en]

def cantidad_palabras(oracion):
    return len([p for p in nltk.word_tokenize(oracion) if p.isalpha()])

# Mismas oraciones candidatas para los dos métodos: entre 10 y 40 palabras
# (descarta exclamaciones sueltas como "America!" y oraciones demasiado largas)
candidatas = [i for i, o in enumerate(oraciones) if 10 <= cantidad_palabras(o) <= 40]
N_ORACIONES = 8

print("Caracteres del discurso:", len(texto_obama))
print("Oraciones totales:", len(oraciones), "| candidatas (10 a 40 palabras):", len(candidatas))

Caracteres del discurso: 13439
Oraciones totales: 110 | candidatas (10 a 40 palabras): 73


### 2.1 Método 1: frecuencia de palabras

In [7]:
frecuencias = Counter(p for o in oraciones for p in palabras_de_contenido(o))
maximo = max(frecuencias.values())
frec_normalizada = {p: c / maximo for p, c in frecuencias.items()}

print("Palabras más frecuentes (sin stop words):")
print(frecuencias.most_common(12))

puntaje_frec = {i: sum(frec_normalizada[p] for p in palabras_de_contenido(oraciones[i])) for i in candidatas}
elegidas_frec = sorted(sorted(puntaje_frec, key=puntaje_frec.get, reverse=True)[:N_ORACIONES])

print("\nRESUMEN 1 — Frecuencia de palabras")
for i in elegidas_frec:
    print(f"[{i}] (score {puntaje_frec[i]:.2f}) {oraciones[i]}")

Palabras más frecuentes (sin stop words):
[('us', 23), ('nation', 12), ('new', 11), ('america', 10), ('every', 8), ('must', 8), ('people', 7), ('less', 7), ('world', 7), ('let', 7), ('today', 6), ('time', 6)]

RESUMEN 1 — Frecuencia de palabras
[14] (score 2.39) Less measurable but no less profound is a sapping of confidence across our land -- a nagging fear that America's decline is inevitable, that the next generation must lower its sights.
[27] (score 2.52) Rather, it has been the risk-takers, the doers, the makers of things'some celebrated but more often men and women obscure in their labor, who have carried us up the long, rugged path towards prosperity and freedom.
[28] (score 2.04) For us, they packed up their few worldly possessions and traveled across oceans in search of a new life.
[63] (score 2.43) They understood that our power alone cannot protect us, nor does it entitle us to do as we please.
[82] (score 2.26) And yet, at this moment -- a moment that will define a generat

### 2.2 Método 2: TextRank

In [8]:
textos_candidatos = [oraciones[i] for i in candidatas]
matriz_tfidf = TfidfVectorizer(stop_words="english").fit_transform(textos_candidatos)
similitud = cosine_similarity(matriz_tfidf)
np.fill_diagonal(similitud, 0)                 # una oración no se vota a sí misma

grafo = nx.from_numpy_array(similitud)
pagerank = nx.pagerank(grafo)                  # posición en la lista de candidatas -> puntaje
puntaje_tr = {candidatas[k]: v for k, v in pagerank.items()}
elegidas_tr = sorted(sorted(puntaje_tr, key=puntaje_tr.get, reverse=True)[:N_ORACIONES])

print("RESUMEN 2 — TextRank")
for i in elegidas_tr:
    print(f"[{i}] (score {puntaje_tr[i]:.4f}) {oraciones[i]}")

RESUMEN 2 — TextRank
[8] (score 0.0223) That we are in the midst of crisis is now well understood.
[10] (score 0.0228) Our economy is badly weakened, a consequence of greed and irresponsibility on the part of some, but also our collective failure to make hard choices and prepare the nation for a new age.
[21] (score 0.0255) We remain a young nation, but in the words of Scripture, the time has come to set aside childish things.
[39] (score 0.0253) Starting today, we must pick ourselves up, dust ourselves off, and begin again the work of remaking America.
[66] (score 0.0238) Guided by these principles once more, we can meet those new threats that demand even greater effort -- even greater cooperation and understanding between nations.
[73] (score 0.0227) To the Muslim world, we seek a new way forward, based on mutual interest and mutual respect.
[76] (score 0.0249) To the people of poor nations, we pledge to work alongside you to make your farms flourish and let clean waters flow; to nou

### 2.3 Comparación lado a lado

In [9]:
comunes = sorted(set(elegidas_frec) & set(elegidas_tr))
print("Oraciones elegidas por ambos métodos:", comunes if comunes else "ninguna")

def temas(indices):
    return Counter(p for i in indices for p in palabras_de_contenido(oraciones[i]))

temas_frec, temas_tr = temas(elegidas_frec), temas(elegidas_tr)
print("Palabras más presentes en el resumen 1:", temas_frec.most_common(8))
print("Palabras más presentes en el resumen 2:", temas_tr.most_common(8))
print("Palabras clave presentes en ambos resúmenes:", sorted(set(temas_frec) & set(temas_tr)))

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame({
    "Frecuencia de palabras": [f"[{i}] {oraciones[i]}" for i in elegidas_frec],
    "TextRank": [f"[{i}] {oraciones[i]}" for i in elegidas_tr],
}))

Oraciones elegidas por ambos métodos: ninguna
Palabras más presentes en el resumen 1: [('us', 8), ('less', 2), ('across', 2), ('generation', 2), ('must', 2), ('rather', 2), ('moment', 2), ('hours', 2)]
Palabras más presentes en el resumen 2: [('new', 3), ('make', 2), ('nation', 2), ('work', 2), ('even', 2), ('greater', 2), ('nations', 2), ('mutual', 2)]
Palabras clave presentes en ambos resúmenes: ['america', 'come', 'let', 'must', 'new', 'traveled', 'understood', 'us', 'words']


,Frecuencia de palabras,TextRank
0,"[14] Less measurable but no less profound is a sapping of confidence across our land -- a nagging fear that America's decline is inevitable, that the next generation must lower its sights.",[8] That we are in the midst of crisis is now well understood.
1,"[27] Rather, it has been the risk-takers, the doers, the makers of things'some celebrated but more often men and women obscure in their labor, who have carried us up the long, rugged path towards prosperity and freedom.","[10] Our economy is badly weakened, a consequence of greed and irresponsibility on the part of some, but also our collective failure to make hard choices and prepare the nation for a new age."
2,"[28] For us, they packed up their few worldly possessions and traveled across oceans in search of a new life.","[21] We remain a young nation, but in the words of Scripture, the time has come to set aside childish things."
3,"[63] They understood that our power alone cannot protect us, nor does it entitle us to do as we please.","[39] Starting today, we must pick ourselves up, dust ourselves off, and begin again the work of remaking America."
4,"[82] And yet, at this moment -- a moment that will define a generation -- it is precisely this spirit that must inhabit us all.","[66] Guided by these principles once more, we can meet those new threats that demand even greater effort -- even greater cooperation and understanding between nations."
5,"[84] It is the kindness to take in a stranger when the levees break, the selflessness of workers who would rather cut their hours than see a friend lose their job which sees us through our darkest hours.","[73] To the Muslim world, we seek a new way forward, based on mutual interest and mutual respect."
6,"[104] In the face of our common dangers, in this winter of our hardship, let us remember these timeless words.","[76] To the people of poor nations, we pledge to work alongside you to make your farms flourish and let clean waters flow; to nourish starved bodies and feed hungry minds."
7,"[105] With hope and virtue, let us brave once more the icy currents, and endure what storms may come.","[96] So let us mark this day with remembrance, of who we are and how far we have traveled."


### 2.4 Comparación de los resúmenes (sobre los resultados obtenidos)

- **Oraciones repetidas:** ninguna. Los dos métodos eligen 8 oraciones totalmente distintas.
- **Coincidencia:** los dos resúmenes arrancan con la **crisis** que vive el país. Frecuencia elige la oración [14] (el miedo a que la decadencia de EE. UU. sea inevitable) y TextRank elige la [8] y la [10] (la crisis y la economía debilitada). También coinciden en la idea de **camino recorrido**: la [28] ("traveled across oceans") en uno y la [96] ("how far we have traveled") en el otro.
- **Qué destaca el resumen por frecuencia:** está dominado por la palabra *us* (aparece 23 veces en el discurso y 8 en el resumen, porque no está en la lista de stop words de NLTK). Por eso prioriza oraciones sobre el **"nosotros" colectivo y el sacrificio**: los trabajadores y migrantes que construyeron el país [27, 28, 84] y el llamado final a resistir "las tormentas" [104, 105].
- **Qué destaca el resumen por TextRank:** elige oraciones que comparten vocabulario con muchas otras, y quedan los ejes "programáticos": **economía y crisis** [8, 10], **renovación del país** [21, 39] y **política exterior** [66, 73, 76], con los mensajes al mundo musulmán y a las naciones pobres, que el otro resumen no menciona.
- **Conclusión:** el método de frecuencia depende mucho de qué palabras se repiten (y de la lista de stop words); TextRank da un resumen más variado en temas porque mide cuánto se relaciona cada oración con el resto.

## Frases para la exposición
1. **Qué hice:** representé los documentos con TF-IDF en tres variantes (texto original, sin stop words y bigramas) para buscar los 3 más parecidos a una consulta con la similitud del coseno, y resumí el discurso de Obama de 2009 con dos métodos extractivos: frecuencia de palabras y TextRank.
2. **Qué cambió al buscar documentos:** *(pendiente hasta cargar los textos del TP 1 y del TP 3).*
3. **Qué observé al comparar los resúmenes:** los dos métodos no repiten ninguna oración, aunque ambos arrancan por la crisis; el de frecuencia se centra en el "nosotros" y el sacrificio colectivo, y TextRank cubre más temas, como la economía, la renovación y la política exterior.